# CS570 – Week 1 Lab: Spark First Contact  
**Notebook:** `week1_spark_intro.ipynb`

## Goal
Understand **how Spark behaves as a system**:
- What is *planned* vs what is *executed*
- Transformations vs actions
- Lazy evaluation and DAGs

> This is **not** a performance lab. We are not tuning Spark or benchmarking runtimes today.


## 0. Setup check (run this first)
This cell confirms your environment is roughly correct.


In [1]:
from pyspark.sql import SparkSession 
spark =SparkSession.builder.master("local[*]").appName("cs570-week1").getOrCreate() 
spark.range(10).count()

10

In [2]:
import sys
import pyspark

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)


Python: 3.11.9
PySpark: 3.5.3


## 1. Create a SparkSession
This is your entry point into Spark from Python.


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("cs570-week1")
    .master("local[*]")   # use all local CPU cores
    .getOrCreate()
)

spark


### Optional: Find the Spark UI link
If the UI is enabled, this prints a URL you can open in your browser.


In [4]:
ui_url = spark.sparkContext.uiWebUrl
print("Spark UI:", ui_url if ui_url else "(Spark UI not available in this environment)")

Spark UI: http://Fsehaye.lan:4040


### Configure display settings

In [6]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 5)
spark.conf.set("spark.sql.debug.maxToStringFields", 100)

# Optional: make pandas previews show all columns too
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)


## 2. Get a dataset
### Generate a small synthetic event dataset  
This avoids download friction and still matches the lecture narrative (events, clicks, logs).


In [7]:
from pathlib import Path
import pandas as pd
import random
import datetime as dt

# Folder where we'll store data next to this notebook
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Synthetic clickstream-like events
random.seed(570)

users = [f"u{n:03d}" for n in range(1, 61)]
pages = ["/", "/search", "/product", "/cart", "/checkout", "/help", "/profile"]
devices = ["mobile", "desktop", "tablet"]
countries = ["US", "DE", "IN", "BR", "CA", "GB"]

rows = []
start = dt.datetime(2026, 1, 1, 9, 0, 0)
for i in range(1200):
    t = start + dt.timedelta(seconds=30*i)
    rows.append({
        "event_time": t.isoformat(),
        "user_id": random.choice(users),
        "page": random.choice(pages),
        "device": random.choice(devices),
        "country": random.choice(countries),
        "dwell_seconds": random.randint(1, 120),
        "is_purchase": 1 if random.random() < 0.04 else 0
    })

pdf = pd.DataFrame(rows)

DATA_PATH = data_dir / "week1_events.csv"
pdf.to_csv(DATA_PATH, index=False)

DATA_PATH.as_posix(), pdf.head()

('data/week1_events.csv',
             event_time user_id      page   device country  dwell_seconds  is_purchase
 0  2026-01-01T09:00:00    u005   /search   mobile      IN              7            1
 1  2026-01-01T09:00:30    u052  /product  desktop      CA             76            0
 2  2026-01-01T09:01:00    u024  /product   mobile      BR            102            0
 3  2026-01-01T09:01:30    u021  /product   tablet      US             92            0
 4  2026-01-01T09:02:00    u029  /product   tablet      GB             68            1)

## 3. Load data into Spark
Key idea: **loading + defining** is not the same as **executing**.


In [8]:
from pyspark.sql.types import *

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH.as_posix())
)

df


event_time,user_id,page,device,country,dwell_seconds,is_purchase
2026-01-01 09:00:00,u005,/search,mobile,IN,7,1
2026-01-01 09:00:30,u052,/product,desktop,CA,76,0
2026-01-01 09:01:00,u024,/product,mobile,BR,102,0
2026-01-01 09:01:30,u021,/product,tablet,US,92,0
2026-01-01 09:02:00,u029,/product,tablet,GB,68,1


In [9]:
df.printSchema()


root
 |-- event_time: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- page: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- dwell_seconds: integer (nullable = true)
 |-- is_purchase: integer (nullable = true)



In [10]:
df.show(5, truncate=False)


+-------------------+-------+--------+-------+-------+-------------+-----------+
|event_time         |user_id|page    |device |country|dwell_seconds|is_purchase|
+-------------------+-------+--------+-------+-------+-------------+-----------+
|2026-01-01 09:00:00|u005   |/search |mobile |IN     |7            |1          |
|2026-01-01 09:00:30|u052   |/product|desktop|CA     |76           |0          |
|2026-01-01 09:01:00|u024   |/product|mobile |BR     |102          |0          |
|2026-01-01 09:01:30|u021   |/product|tablet |US     |92           |0          |
|2026-01-01 09:02:00|u029   |/product|tablet |GB     |68           |1          |
+-------------------+-------+--------+-------+-------+-------------+-----------+
only showing top 5 rows



## 4. Transformations (define work, don’t execute yet)
Try a few transformations. These should feel “instant” because Spark is *building a plan*.


In [13]:
from pyspark.sql import functions as F

# Example: filter + select + derived column
df_t = (
    df
    .filter(F.col("country") == "US")
    .select("event_time", "user_id", "page", "device", "country", "dwell_seconds", "is_purchase")
    .withColumn("is_high_intent", (F.col("page").isin(["/cart", "/checkout"])).cast("int"))
)

df_t


event_time,user_id,page,device,country,dwell_seconds,is_purchase,is_high_intent
2026-01-01 09:01:30,u021,/product,tablet,US,92,0,0
2026-01-01 09:04:30,u052,/cart,desktop,US,5,0,1
2026-01-01 09:06:30,u015,/cart,mobile,US,47,0,1
2026-01-01 09:12:00,u043,/profile,desktop,US,98,0,0
2026-01-01 09:13:00,u048,/checkout,tablet,US,29,0,1


### Inspect the logical plan (no execution yet)
This shows the **plan** Spark has built.


In [12]:
df_t.explain(True)


== Parsed Logical Plan ==
'Project [event_time#25, user_id#26, page#27, device#28, country#29, dwell_seconds#30, is_purchase#31, cast('page IN (/cart,/checkout) as int) AS is_high_intent#155]
+- Project [event_time#25, user_id#26, page#27, device#28, country#29, dwell_seconds#30, is_purchase#31]
   +- Filter (country#29 = US)
      +- Relation [event_time#25,user_id#26,page#27,device#28,country#29,dwell_seconds#30,is_purchase#31] csv

== Analyzed Logical Plan ==
event_time: timestamp, user_id: string, page: string, device: string, country: string, dwell_seconds: int, is_purchase: int, is_high_intent: int
Project [event_time#25, user_id#26, page#27, device#28, country#29, dwell_seconds#30, is_purchase#31, cast(page#27 IN (/cart,/checkout) as int) AS is_high_intent#155]
+- Project [event_time#25, user_id#26, page#27, device#28, country#29, dwell_seconds#30, is_purchase#31]
   +- Filter (country#29 = US)
      +- Relation [event_time#25,user_id#26,page#27,device#28,country#29,dwell_second

## 5. Actions (trigger execution)
Now run an action. This is where Spark *actually does work*.


In [18]:
df_t.count()


204

In [15]:
# Simple aggregation (GROUP BY)
df_g = (
    df.groupBy("country")
      .agg(
          F.count("*").alias("num_events"),
          F.sum("is_purchase").alias("num_purchases"),
          F.avg("dwell_seconds").alias("avg_dwell_seconds")
      )
      .orderBy(F.desc("num_events"))
)

df_g.show(truncate=False)


+-------+----------+-------------+------------------+
|country|num_events|num_purchases|avg_dwell_seconds |
+-------+----------+-------------+------------------+
|GB     |210       |7            |59.642857142857146|
|US     |204       |8            |57.03921568627451 |
|DE     |200       |14           |60.095            |
|BR     |197       |11           |59.401015228426395|
|CA     |195       |8            |58.697435897435895|
|IN     |194       |5            |61.74742268041237 |
+-------+----------+-------------+------------------+



## 6. Connect back to lecture
Before we wrap up, think about these questions:
1. What is the difference between a **transformation** and an **action**?
2. What does **lazy evaluation** mean in Spark?
3. What surprised you most about Spark's behavior?


## 7. (Optional) Small experiment
Change the filter condition and rerun the action.  
Predict what will happen before you run it.


In [17]:
df_t2 = df.filter(F.col("device") == "mobile").select("user_id", "page", "device", "country", "is_purchase")

# Predict: Will this do any work yet?
df_t2.explain()

# Now trigger execution
df_t2.count()


== Physical Plan ==
*(1) Filter (isnotnull(device#28) AND (device#28 = mobile))
+- FileScan csv [user_id#26,page#27,device#28,country#29,is_purchase#31] Batched: false, DataFilters: [isnotnull(device#28), (device#28 = mobile)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/d:/SFBU/Spring_semester_2026/CS570/week-1/data/week1_events.csv], PartitionFilters: [], PushedFilters: [IsNotNull(device), EqualTo(device,mobile)], ReadSchema: struct<user_id:string,page:string,device:string,country:string,is_purchase:int>




407